# Project 2: modellen (Jeff)

Stap 3: embed-and-pool met GloVe Twitter. Zelfde folds en seeds als project 1 (5 folds x 3 seeds), macro-F1.

## Setup

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)
root = Path.cwd().parent
sys.path.insert(0, str(root))

from huggingface_hub.utils import logging as hf_logging
hf_logging.set_verbosity_error()  # geen HF_TOKEN-melding bij het laden van de data

from part2.clslib import PretrainedEmbeddings
from src.data import LABEL_NAMES, load_stance_all, make_cv_splits
from src.embeddings import write_subset
from src.evaluate import compare_pairs, run_cross_validation
from src.features import ENCODER_FACTORIES
from src.glove_tokenizer import GloveTwitterTokenizer, tokenize
from src.models import MODEL_RUNNERS
from src.neural import cross_validate, silenced
from src.plots import plot_bayes_comparisons

DATA = root / "data"
MODELS_DIR = DATA / "models"  # data/ staat in .gitignore

## Data en GloVe Twitter

In [ ]:
stance_dataset = load_stance_all()
cv_runs = make_cv_splits(stance_dataset)
n_runs = len(cv_runs)

# alleen de woorden die in onze tweets voorkomen (alle splits) uit GloVe halen: sneller en kleine modelbestanden
words = {t for split in stance_dataset for text in stance_dataset[split]["text"] for t in tokenize(text)}

def load_glove(dims):
    subset = DATA / f"glove.twitter.27B.{dims}d.stance.txt"
    if not subset.is_file():
        write_subset(DATA / f"glove.twitter.27B.{dims}d.txt", subset, words)
    return PretrainedEmbeddings(str(subset), dims)

glove = {dims: load_glove(dims) for dims in (50, 100)}
print(f"{len(words)} verschillende tokens in onze data, {len(glove[50].vocab) - 2} daarvan in GloVe Twitter")

## Stap 3: embed-and-pool

Tweet → GloVe-vector per token → gemiddelde (of som) → MLP met 2 lagen → klasse.
Ontwerpkeuzes die we testen: embeddings vast of trainbaar, gemiddelde of som, 50 of 100 dimensies.

## Stap 3: embed-and-pool

Tweet → GloVe-vector per token → gemiddelde (of som) → MLP met 2 lagen → klasse.

| Opzet | Wat verandert t.o.v. `pool_50d` |
|---|---|
| `pool_50d` | basis: 50d, embeddings vast, gemiddelde |
| `pool_50d_trainable` | embeddings worden mee-getraind |
| `pool_50d_sum` | som i.p.v. gemiddelde |
| `pool_100d` | 100 dimensies |

In [ ]:
%%capture
REGRESSOR = {"hidden_size1": 50, "hidden_size2": 50, "dropout": 0.}
OPTIMISER = {"learning_rate": 1e-3, "weight_decay": 1e-4, "batch_size": 64, "num_epochs": 20}

configs = {
    "pool_50d":           dict(dims=50,  freeze_embeddings=True,  average_pool=True),
    "pool_50d_trainable": dict(dims=50,  freeze_embeddings=False, average_pool=True),
    "pool_50d_sum":       dict(dims=50,  freeze_embeddings=True,  average_pool=False),
    "pool_100d":          dict(dims=100, freeze_embeddings=True,  average_pool=True),
}

f1_scores = {}
for name, cfg in configs.items():
    emb = glove[cfg["dims"]]
    f1_scores[name] = cross_validate(
        cv_runs, GloveTwitterTokenizer(emb), emb, "embed-and-pool",
        enc_kwargs={"freeze_embeddings": cfg["freeze_embeddings"], "average_pool": cfg["average_pool"]},
        regressor_kwargs=REGRESSOR, opt_kwargs=OPTIMISER, save_to=MODELS_DIR / name,
    )

# ter vergelijking: beste systeem van project 1, opnieuw gedraaid op dezelfde folds
with silenced():
    f1_scores.update(run_cross_validation(
        cv_runs, {"tfidf": ENCODER_FACTORIES["tfidf"]}, {"mlp_regularized": MODEL_RUNNERS["mlp_regularized"]},
        num_classes=len(LABEL_NAMES),
    ))

### Resultaten (macro-F1, 15 runs)

In [ ]:
pd.DataFrame({
    "gemiddelde": {name: np.mean(s) for name, s in f1_scores.items()},
    "std": {name: np.std(s) for name, s in f1_scores.items()},
}).sort_values("gemiddelde", ascending=False).round(3)

### Welke ontwerpkeuzes maken verschil?

In [ ]:
best_pool = max(configs, key=lambda name: np.mean(f1_scores[name]))
pairs = [("pool_50d", name) for name in configs if name != "pool_50d"]
pairs.append((best_pool, "mlp_regularized_tfidf"))

comparison = compare_pairs(f1_scores, pairs, runs=n_runs)
plot_bayes_comparisons(comparison, "Design choices (vs pool_50d) and best pool vs project 1");